In [ ]:
"""Inserts"""

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import time as tm
from typing import Any, Callable
import warnings
warnings.filterwarnings("ignore")

## I. First Model: KNN Method

### 1. Missing Values / Outliers managers

In [ ]:
def window_cleaning(rows: pd.DataFrame, margin: float = 1.5,window : int = 50) -> pd.DataFrame:

    """Cleans a set of rows by computing the median, and cutting the values under and above a
    certain distance of it (corresponding to margin-times the standart error) and replacing
    these values by the median.
    Args:
        rows: the rows to clean.
        margin: coefficient of the standart error to create an interval of "okay" values.
    Returns:
        rows: same rows with replaced outliers.
    """
    
    n=rows.shape[0]
    rows.reset_index(inplace=True)
    for t in range(window, n+window , window):
        
        if t >= n:
            t = n
        
        mask = ((t-window)*864000 <= rows["time"])*( rows["time"] < t*864000)
        window_rows = rows.loc[mask]
        
        med = np.median(window_rows.dropna()["temperature"])
        std_error = np.std(window_rows.dropna()["temperature"])
        lb = med - margin*std_error
        ub = med + margin*std_error
        
        outlier_index = window_rows.index[
        (window_rows["temperature"] < lb) | 
        (window_rows["temperature"] > ub) | 
        (window_rows["temperature"].isna())]
        
        rows.loc[outlier_index, "temperature"] = med
        
    return rows


def preprocessing(sensors_set: pd.DataFrame, window: int = 50, margin: float = 1.5) -> pd.DataFrame:

    """The global function to preprocess the Data. It cleans each sensor one by one by looking
    at 'windows' of time values on their data, computing the median on such intervals, and
    replacing too-far values of the median by itself.
    Args:
        sensors_set: dataset of all sensors to preprocess, with values of time,
         power and temperature.
        window: amount of rows to take to compute the median.
        margin: size of the safe zone, where the values are kept around the median.
    Returns:
        sensors_set: with cleaner data
    """
    dropped = []

    
    for sample in sensors_set.index.unique(): # We pick every specific sensor of the dataset
        if sample[0] in dropped:
            continue
        all_rows = sensors_set.loc[sample] # All the temperatures of a given sample and power tier
        if np.mean(all_rows.dropna()["temperature"][:-5:-1])< 0 or np.mean(all_rows["temperature"][:-5:-1])>25:
            sensors_set.drop(sample[0],inplace=True)
            print(f"Sample {sample[0]} dropped")
            dropped.append (sample[0])
            continue
        
        cleaned = window_cleaning(rows=all_rows, margin=margin).set_index(["sensor","tier"])
        sensors_set.loc[sample] = cleaned

    """"""
    return sensors_set

### 2. Distance Metrics

In [ ]:
def man_dist(sample_coo: pd.DataFrame, train_set_coo: pd.DataFrame) -> pd.DataFrame:

    """Computes the Manhattan distance between a sample and all train sensors, and add
    their values in a new column of train_set_coo.
    Args:
        sample_coo: Dataset of the sample's coordinates
        train_set_coo: Dataset of the sensors' coordinates
    Returns:
        train_set_coo_with_dist: train_set_coo with a column "distance", filled.
    """

    coor_train = np.array([train_set_coo["coor_x"], train_set_coo["coor_y"]]).T
    coor_sample = np.array([sample_coo["coor_x"], sample_coo["coor_y"]])

    train_set_coo["distance"] = np.abs(coor_train - coor_sample).sum(axis = 1)

    return train_set_coo
   


def eucli_dist(sample_coo: pd.DataFrame, train_set_coo: pd.DataFrame) -> pd.DataFrame:

    """Computes the Euclidian distance between a sample and all train sensors, and add
    their values in a new column of train_set_coo.
    Args:
        sample_coo: Dataset of the sample's coordinates
        train_set_coo: Dataset of the sensors' coordinates
    Returns:
        train_set_coo_with_dist: train_set_coo with a column "distance", filled.
    """
    
    coor_train = np.array([train_set_coo["coor_x"], train_set_coo["coor_y"]]).T
    coor_sample = np.array([sample_coo["coor_x"], sample_coo["coor_y"]])

    train_set_coo["distance"] = np.sqrt(((coor_train - coor_sample) ** 2).sum(axis = 1))

    return train_set_coo


### 3. Distance Score

In [ ]:
def distance_score(neighbors:pd.DataFrame,
                   score_parameter: float = 0.1) -> pd.DataFrame:
    
    """Replace the "distance" column by a "score" one to each given neighbor sensor,
    depending on their distance to the sample.
    The closer is the sample, the higher the score. We set an upper limit to it
    to avoid giving "blind confidence" to "very close" sensors.
    Args:
        neighbors: neighbor sensors, with "distance" column, regarding the sample.
        score_parameter: Hyper-parameter, set to 1 by default.
    Returns:
        neighbors: Updated with the "score" column.
    """

    scores = 1 / (score_parameter * neighbors["distance"] + 1)

    # We replace the distance column (which we don't need anymore) by the "score" one.
    neighbors["score"] = scores
    neighbors.drop(labels = ["distance"], axis = 1, inplace = True)
    
    return neighbors

### 4. K-Nearest Neighbors

In [ ]:
def find_nearest_neighbors(
    sample_coo: pd.DataFrame, 
    train_set_coo: pd.DataFrame, 
    distance_fn: Callable = eucli_dist, 
    k: int = 7,
    score_parameter: int = 1) -> pd.DataFrame:
    
    """Finds the names and distance-scores of the k-Nearest Neighbors of one sample.
    Args:
        sample_coo: Coordinates of the sample.
        train_set_coo: Dataset of the the sensors' coordinates.
        distance_fn: Distance function.
        k: Number of nearest neighbors considered, set to 7 by default.
        score_parameter: Hyper-parameter of distance score, set to 1 by default.
    Returns:
        neighbors: dataframe of the k-nearest neighbors of the sample, with their distance's scores
        stored in a new "score" column.
    """

    train_set_with_dist = distance_fn(sample_coo, train_set_coo) # We compute the distances of sample to all sensors 
    neighbors = distance_score(neighbors= train_set_with_dist.sort_values(by=["distance"]).head(k),
                               score_parameter= score_parameter) # We select the closest ones, and replace distance by score
    
    return neighbors

### 5. Prediction Functions

In [ ]:
def prediction_single(
    
    time2pred : np.ndarray,
    neighbors_values: pd.DataFrame,
    neighbors_scores: pd.DataFrame,
    k: int = 7) -> list:

    """Gives the final prediction of a given sample's temperature, while calling KNN. Calculates the
    mean value of the temperatures of the neighbors, weighted by their distance scores.
    Args:
        neighbors_values: The dataframe of values of the k-neighbors of the sample.
        neighbors_scores: The dataframe of scores (and coordinates) of the neighbors.
        k: Number of nearest neighbors taken, set to 7 by default

    Returns:
        pred: The values of the predicted temperatures of the sample, for all times and powers.
    """

    pred = []
    total_scores = np.sum(neighbors_scores["score"])
    
    for pow_tier in range(3):
        rows = neighbors_values[neighbors_values["tier"] == pow_tier]
        for time in time2pred:
            mask = (rows.time == time)
            temperatures = rows.loc[mask, "temperature"]
            pred.append(np.sum(temperatures * neighbors_scores["score"]) / total_scores)

    return pred



def prediction(
    validation_set_coo: pd.DataFrame,
    validation_set_values:pd.DataFrame,
    train_set_coo: pd.DataFrame,
    train_set_values: pd.DataFrame,
    distance_fn: Callable = eucli_dist, 
    k: int = 7,
    score_parameter: int = 1) -> pd.DataFrame:

    """Gives the final predictions of all validation-set temperatures, while calling prediction_single.
    Args:
        validation_set_coo: Dataframe with names and coordinates of some sensors whom we "hide" the
         temperature values.
        validation_set_values: Dataframe with names, times and powers of the validation set at which we want the
         model to evaluate the temperature (times with "not nan" temperatures)
        train_set_coo: Dataset of the "train" sensors' coordinates.
        train_set_values: Dataset of the "train" sensors' values of time, power and temperature.
        distance_fn: Distance function (among Manhattan or Euclidian)
        k: Number of nearest neighbors taken, set to 7 by default
        score_parameter: Hyper-parameter of distance score, set to 1 by default.

    Returns:
        pred: validation_set_values with a column "prediction", filled with the outputs of prediction_single.
    """

    # Empty list in which we'll store our predictions
    validation_set_values["prediction"] =  np.empty(validation_set_values.shape[0])
    time2pred = validation_set_values["time"].unique()
    for sensor_name, row in validation_set_coo.iterrows():
        start = tm.time()
        
        neighbors = find_nearest_neighbors(sample_coo = row,
                                           train_set_coo = train_set_coo,
                                           distance_fn = distance_fn,
                                           k = k, 
                                           score_parameter=score_parameter)
        
        validation_set_values.loc[sensor_name,"prediction"] = prediction_single(
                                                   time2pred=time2pred ,
                                                   neighbors_values= train_set_values.loc[neighbors.index],
                                                   neighbors_scores= neighbors,
                                                   k = k)
        end = tm.time()
        print(f"Prediction de {sensor_name} terminée. Temps: {end - start}")



    # Adding the column "prediction" to our validation set
    
    return validation_set_values

### 6. Loss Function

In [ ]:
def loss(ground_truths: pd.DataFrame, prediction_set: pd.DataFrame) -> float:
    
    """ Computes the mean error of the model.
    Args:
        ground_truths: Dataframe of the name of the sensors with their time, and real temperatures.
        prediction_set: Our "Prediction" dataframe, of the same structure as ground_truths, but with
        a column "prediction" instead of "temperature".
    Returns:
        float: The loss
    """
    
    real_temp = ground_truths["temperature"]
    predictions = prediction_set["prediction"]

    return np.mean(np.abs(real_temp - predictions))

### Training Session

In [ ]:
# Reading data
training_data=pd.read_parquet("data_parquet_2026/train.parquet")
sensors = pd.read_parquet("data_parquet_2026/sensors.parquet").drop(columns=["coor_z"]).drop_duplicates().set_index(keys= "sensor")
test_data = pd.read_parquet("data_parquet_2026/test.parquet").set_index(keys= "sensor")

training_data.set_index("sensor",inplace=True)
training_data["tier"] = np.empty(training_data.shape[0],dtype=int)
n=int(len(training_data["power"])/len(training_data.index.unique())/3) #27384/3 = 9128
for senso in training_data.index.unique():
    stack = [0]*n + [1]*n + [2]*n    
    tesvar=training_data.loc[senso,"temperature"]
    training_data.loc[senso, "tier"] = stack

In [ ]:
"""Preprocess the data"""
training_data.plot.scatter(x="time",y="temperature",alpha=0.5)

print("Starting preprocess")

training_data.reset_index(inplace=True)
training_data.set_index(["sensor","tier"],inplace=True)

training_data = preprocessing(training_data)

print("first done")


training_data.plot.scatter(x="time",y="temperature",alpha=0.5)

training_data.reset_index(inplace=True)

p1=training_data[training_data["tier"]==0]
p2=training_data[training_data["tier"]==1]
p3=training_data[training_data["tier"]==2]

p1.plot.scatter(x="time",y="temperature",alpha=0.5)
p2.plot.scatter(x="time",y="temperature",alpha=0.5)
p3.plot.scatter(x="time",y="temperature",alpha=0.5)


In [ ]:
"""Dropping the test sensors, as we will train the model only with the training data"""
test_sensors = test_data.index.unique()

training_data.reset_index(inplace=True)
training_data.set_index("sensor",inplace=True)

train_sensors = sensors.loc[training_data.index.unique()]

""" A Garder pour d autres tests
Sampling for TESTING ON SMALLER SETS"""
#train_sensors = train_sensors.sample(frac = 1/5) # 20% des sensors
#training_data = training_data.loc[train_sensors.index].query('time > 5e9') # 10% des times
""""""


"""Partition of the training_data into our "training" and "validation" model's sets""" 
# Validation/ train coordinates' sets partition, over the train_sensors_set
validation_sensors_coo = train_sensors.sample(frac = 1/5)
train_sensors_coo = train_sensors.drop(labels=validation_sensors_coo.index)

# Validation/ train values' sets partition, ignoring unknown data of validation,
# and replacing them in training set
validation_samples_with_temp = training_data.loc[validation_sensors_coo.index]
train_samples_with_temp = training_data.drop(validation_sensors_coo.index)

In [ ]:
"""Let's train our model!"""
losses=[]
ks=[]

for k in range(12,20):
    validation_samples_with_pred = prediction(
        validation_set_coo = validation_sensors_coo,
        validation_set_values = validation_samples_with_temp,
        train_set_coo = train_sensors_coo,
        train_set_values = train_samples_with_temp,
        distance_fn = eucli_dist, 
        k = k)
    
    

    """Let's evaluate it!"""
    get_loss = loss(
        ground_truths = validation_samples_with_temp.dropna(),
        prediction_set= validation_samples_with_pred.dropna())
    ks.append(k)
    losses.append(get_loss)
    print(f"Value of k: {k} \n Loss: {get_loss} \n --------- \n")
    
plt.scatter(ks,losses)

In [ ]:
"""lets try"""
sub = prediction(
        validation_set_coo = sensors.loc[test_sensors],
        validation_set_values = test_data,
        train_set_coo = sensors.loc[train_sensors.index],
        train_set_values = training_data,
        distance_fn = eucli_dist, 
        k = 12)

In [ ]:

submission = pd.DataFrame({
   "Id": np.arange(len(test_data), dtype=int),
   "temperature": sub["prediction"],
})
assert list(submission.columns) == ["Id", "temperature"]
assert len(submission) == len(test_data)
assert (submission["Id"].to_numpy() == np.arange(len(test_data))).all()
assert np.isfinite(submission["temperature"]).all()
assert submission.isna().sum().sum() == 0
submission.to_csv("submission.csv", index=False)
print("Saved submission.csv")

#train_samples_with_temp.query('time < 1e9').plot.scatter(x="time",y="temperature",alpha=0.5)

## II. A few improvements

### 1. Outliers Manager V2

In [ ]:
"""NOTE: We may clean multiple times same sensors but we don't care (multiple times neighbors)"""

def preprocessing_with_KNN(sensors_set: pd.DataFrame, sensors_coo: pd.DataFrame, k: int = 10, margin: float = 10) -> pd.DataFrame:

    """The global function to preprocess the Data. It first calls first_clean to delete
    'extreme' outliers, than clean the dataset by looking at the k-neighbors of a sample,
    and computing the median of their temperatures at given time to determine wether to
    keep or not the sample and neighbors' temperature.
    Args:
        sensors_set: dataset of all sensors to preprocess, with values of time,
         power and temperature.
        sensors_coo: dataset of the sensors' coordinates
        k: number of nearest neighbors considered, set to 10 by default
        margin: size of the safe zone, where the values are kept around the median.
    Returns:
        sensors_set: with cleaner data
    """

    # We will check if each sample taken was a former neighbor of another, so we pass it.
    already_clean = np.array([])


    for sample in sensors_set["sensor"].unique(): # We pick every specific sensor of the dataset

        if sample in already_clean:
            continue

        # Here, the sample is counted as its own neighbor (distance == 0)
        neighbors = find_nearest_neighbors(
            sample_coo = sensors_coo["sensor" == sample], 
            train_set_coo = sensors_coo, 
            distance_fn = eucli_dist, 
            k = k + 1)

        ... # Add a line to pick each peculiar power

        for t in range(sensors_set["sensor" == sample].shape[0]):
            rows = sensors_set["sensor" == neighbors.sensor and "time" == t*864000]  # and "power" == power
            rows = window_cleaning(rows=rows, margin=margin)
        
        already_clean = np.concatenate((already_clean, neighbors["sensor"]), axis = 0)
        already_clean = np.unique(already_clean) # Deletes copies

    return sensors_set

### 2. n-Folder Validation
